In [1]:
#import packages

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from scipy import stats
from datetime import datetime as dt
from datetime import date
from ib_insync import *
import os
from google.cloud import storage
from google.cloud import bigquery
import calendar as cld
import nest_asyncio
import mysql.connector
import pyarrow
nest_asyncio.apply()

In [2]:
#sql server info

mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="1Pointer!",
  database="ib_data",
  auth_plugin = 'mysql_native_password'
)

print("Connection Established")

#Connection to IB

ib = IB()

ib.connect('127.0.0.1', 4001, clientId=1)


Connection Established


<IB connected to 127.0.0.1:4001 clientId=1>

Error 200, reqId 185: No security definition has been found for the request, contract: Stock(symbol='BBU.UN', exchange='SMART', primaryExchange='TSE', currency='CAD')
Error 200, reqId 191: No security definition has been found for the request, contract: Stock(symbol='CXB', exchange='SMART', primaryExchange='TSE', currency='CAD')
Error 200, reqId 200: No security definition has been found for the request, contract: Stock(symbol='CWB', exchange='SMART', primaryExchange='TSE', currency='CAD')
Error 200, reqId 212: No security definition has been found for the request, contract: Stock(symbol='CIX', exchange='SMART', primaryExchange='TSE', currency='CAD')
Error 200, reqId 236: No security definition has been found for the request, contract: Stock(symbol='FIL', exchange='SMART', primaryExchange='TSE', currency='CAD')
Error 200, reqId 261: No security definition has been found for the request, contract: Stock(symbol='INE', exchange='SMART', primaryExchange='TSE', currency='CAD')
Error 200, re

In [3]:
#current SPX/TSX constituents
tickercad = ['XMV' ,	'XMI',	'XML',	'XIN',	'GCNS',	'XMS',	'XTOH',	'XMY',	'XEM',	'GGRO',	'GEQT',	'XSC',	'XMM',	'GBAL',	'XEC',	'XUS',	'XEF',	'XMH',	'XSE',	'XMC',	'XSMB',	'CMR',	'CLG',	'CBH',	'CLF',	'CBO',	'XGGB',	'CVD',	'XQB',	'XDIV',	'XMU',	'XQQ',	'XWD',	'XGRO',	'XDUH',	'XBAL',	'XAGG',	'XDG',	'XSU',	'XDU',	'XSUS',	'XSEA',	'XCBG',	'XSHG',	'XDGH',	'XESG',	'XAGH',	'XSTB',	'XGI',	'XCD',	'XFLB',	'XFLI',	'XSEM',	'XSP',	'XFLX',	'XSAB',	'CWO',	'XTLH',	'CRQ',	'XTLT',	'XID',	'FIE',	'XCH',	'XEMC',	'XHC',	'XFR',	'XGB',	'XCB',	'XSB',	'XSI',	'XRB',	'XLB',	'XHB',	'XTR',	'XDRV',	'XBB',	'XSH',	'CWW',	'XCV',	'XCG',	'XUSR',	'XDV',	'XDSR',	'XEU',	'CEW',	'XSTH',	'XEH',	'XCBU',	'XSTP',	'XIGS',	'XSHU',	'COW',	'CIF',	'CGR',	'CYH',	'XDNA',	'XCLN',	'XQQU',	'XEXP',	'XHAK',	'XETM',	'XCHP',	'CIE',	'XUSF',	'XAD',	'XEN',	'XEB',	'CUD',	'CDZ',	'XTOT',	'XQLT',	'XIU',	'CJP',	'XEG',	'XST',	'XIC',	'CPD',	'XSMC',	'XRE',	'XMA',	'XUSC',	'XSMH',	'XFH',	'XIT',	'XFN',	'XMTM',	'XBM',	'XEI',	'XVLU',	'XMD',	'XUT',	'XCSR',	'XCNS',	'XAW',	'XPF',	'XHU',	'XGD',	'XSPC',	'XEQT',	'XUU',	'XINC',	'XUH',	'XCS',	'XIG',	'XHY',	'XHD',	'CLU',	'XMW', 'AAV','AEM',	'AC',	'AGI',	'ASTL',	'AQN',	'ATD',	'AP.UN',	'ALA',	'AIF',	'ARX',	'ATZ',	'ACO.X',	'ATH',	'ATRL',	'ATS',	'AYA',	'BTO',	'BDGI',	'BMO',	'BNS',	'ABX',	'BHC',	'BTE',	'BCE',	'BIR',	'BDT',	'BB',	'BEI.UN',	'BBD.B',	'BLX',	'BYD',	'BAM',	'BBU.UN',	'BN',	'BIP.UN',	'BEP.UN',	'DOO',	'CAE',	'CXB',	'CCO',	'CAR.UN',	'CM',	'CNR',	'CNQ',	'CP',	'CTC.A',	'CU',	'CWB',	'CPX',	'CS',	'CJT',	'CCL.B',	'CLS',	'CVE',	'CG',	'CEU',	'GIB.A',	'CSH.UN',	'CHP.UN',	'CIX',	'CCA',	'CIGI',	'CSU',	'CRR.UN',	'CRT.UN',	'DFY',	'DML',	'DSG',	'DOL',	'DIR.UN',	'DPM',	'ELD',	'EFN',	'EMA',	'EMP.A',	'ENB',	'EFR',	'ENGH',	'EQB',	'EQX',	'ERO',	'EIF',	'FFH',	'FIL',	'FTT',	'FCR.UN',	'AG',	'FM',	'FSV',	'FTS',	'FVI',	'FNV',	'FRU',	'WN',	'GFL',	'GEI',	'GIL',	'GSY',	'GRT.UN',	'GWO',	'HR.UN',	'HWX',	'HBM',	'H',	'IAG',	'IMG',	'IGM',	'IMO',	'INE',	'IFC',	'IFP',	'IPCO',	'IIP.UN',	'IVN',	'JWEL',	'KNT',	'KEL',	'KEY',	'KMP.UN',	'KXS',	'K',	'LIF',	'LB',	'LSPD',	'LNR',	'L',	'LUG',	'LUN',	'MAG',	'MG',	'MFC',	'MFI',	'MATR',	'MDA',	'MEG',	'MX',	'MRU',	'MTY',	'MTL',	'NA',	'NGD',	'NXE',	'NFI',	'NWC',	'NPI',	'NWH.UN',	'NG',	'NTR',	'NVEI',	'NVA',	'OGC',	'ONEX',	'OTEX',	'OLA',	'OR',	'OSK',	'PAAS',	'POU',	'PXT',	'PKI',	'PSI',	'PPL',	'PET',	'PEY',	'POW',	'PSK',	'PD',	'PBH',	'PMZ.UN',	'PRMW',	'QBR.B',	'QSR',	'RCH',	'REI.UN',	'RCI.B',	'RUS',	'SSL',	'SAP',	'SEA',	'SES',	'SHOP',	'SIA',	'SIL',	'SRU.UN',	'SOBO',	'TOY',	'SII',	'SSRM',	'STN',	'STLC',	'SJ',	'SVI',	'SLF',	'SU',	'SPB',	'TVE',	'TRP',	'TECK.B',	'T',	'TFII',	'TRI',	'TLRY',	'X',	'TPZ',	'TXG',	'TIH',	'TD',	'TOU',	'TA',	'TCL.A',	'TFPM',	'TSU',	'VRN',	'VET',	'WCN',	'WDO',	'WFG',	'WPM',	'WCP',	'WPK',	'WSP']

stocklist = []
for t in tickercad:
    try:
        contract = Stock(t,  'SMART', 'CAD', primaryExchange='TSE')
    except:
        contract = Stock(t,  'SMART', 'CAD', primaryExchange='TSE')

    bars = ib.reqHistoricalData(
        contract, endDateTime = '20260424 15:59:00', durationStr='1 D',
        barSizeSetting='1 day', whatToShow='TRADES', useRTH=True, formatDate=1)

    df = util.df(bars)
    # df['date'] = df['date'].dt.tz_convert("America/New_York")
    if df is None:
        print(f"Ticker {t} is empty")
    else:
        df['actual_date'] = df['date']
        df['Ticker'] = t
        print(df.tail(1))
        stocklist.append(df)

#reset index to form one full, single table
stocklist=pd.concat(stocklist,ignore_index=True)
stocklist.reset_index(inplace=True,drop=True)  

#add dates and alter dtypes
stocklist['barCount'] = stocklist['barCount'].astype(float)
stocklist['MarketDate'] = stocklist['date']
stocklist['MarketTime'] = stocklist['date']
stocklist['RegistryDate'] = date.today()

#reorder columns
stocklist = stocklist[['MarketTime','open','high','low','close','volume','average','barCount','MarketDate','Ticker','RegistryDate']]
stocklist['Currency'] = 'CAD'

#Remove date and replace by MarketTime
stocklist['MarketTime'] = stocklist['MarketDate']
stocklist = stocklist[['MarketTime','open','high','low','close','volume','average','barCount','MarketDate','Ticker','RegistryDate','Currency']]

#environment variable
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'c:\\Users\\Johnny\\Desktop\\Repo\\Projects\\financialdata-464612-483d8c5985f1.json'
client = bigquery.Client()

dataset_id = "PriceData"
table_id = "DailyPriceData"
table_ref = client.dataset(dataset_id).table(table_id)
table = client.get_table(table_ref)

job_config = bigquery.LoadJobConfig(
    # schema=schema, # Uncomment if you define a schema
    write_disposition="WRITE_APPEND",  # Overwrite table, or "WRITE_APPEND" to append
)

try:
    # Load DataFrame to BigQuery
    job = client.load_table_from_dataframe(stocklist, table_ref, job_config=job_config)
    job.result()  # Wait for the job to complete

    print(f"Loaded {job.output_rows} rows into {dataset_id}.{table_id}")

except Exception as e:
    print(f"Error loading data to BigQuery: {e}")

         date   open   high    low  close  volume  average  barCount  \
0  2026-04-24  57.71  57.71  57.54  57.66  9635.0   57.608        26   

  actual_date Ticker  
0  2026-04-24    XMV  
         date   open   high   low  close  volume  average  barCount  \
0  2026-04-24  47.43  47.48  47.4  47.43  2956.0   47.431        10   

  actual_date Ticker  
0  2026-04-24    XMI  
         date   open   high    low  close  volume  average  barCount  \
0  2026-04-24  33.02  33.03  33.02  33.03  1000.0   33.022         3   

  actual_date Ticker  
0  2026-04-24    XML  
         date   open   high   low  close   volume  average  barCount  \
0  2026-04-24  44.24  44.46  44.2  44.37  49008.0   44.332       140   

  actual_date Ticker  
0  2026-04-24    XIN  
         date   open   high    low  close  volume  average  barCount  \
0  2026-04-24  49.75  49.75  49.66  49.71  1093.0   49.662         5   

  actual_date Ticker  
0  2026-04-24   GCNS  
         date   open   high    low  close  volu

C:\Users\Johnny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 351 rows into PriceData.DailyPriceData


In [4]:
#current SP500 constituents
ticker = ['TSLA','NVDA','AAPL','MSFT','GOOG','META','A','ABBV','ABNB','ABT',	'ACGL',	'ACN',	'ADBE',	'ADI',	'ADM',	'ADP',	'ADSK',	'AEE',	'AEP',	'AES',	'AFL',	'AIG',	'AIZ',	'AJG',	'AKAM',	'ALB',	'ALGN',	'ALL',	'ALLE',	'AMAT',	'AMCR',	'AMD',	'AME',	'AMGN',	'AMP',	'AMT',		'ANET',	'ANSS',	'AON',	'AOS',	'APA',	'APD',	'APH',	'APO',	'APTV',	'ARE',	'ATO',	'AVB',	'AVGO',	'AVY',	'AWK',	'AXON',	'AXP',	'AZO',	'BA',	'BAC',	'BALL',	'BAX',	'BBY',	'BDX',	'BEN',	'BF.B',	'BG',	'BIIB',	'BK',	'BKNG',	'BKR',	'BLDR',	'BLK',	'BMY',	'BR',	'BRK.B',	'BRO',	'BSX',	'BX',	'BXP',	'C',	'CAG',	'CAH',	'CARR',	'CAT',	'CB',	'CBOE',	'CBRE',	'CCI',	'CCL',	'CDNS',	'CDW',	'CEG',	'CF',	'CFG',	'CHD',	'CHRW',	'CHTR',	'CI',	'CINF',	'CL',	'CLX',	'CMCSA',	'CME',	'CMG',	'CMI',	'CMS',	'CNC',	'CNP',	'COF',	'COIN',	'COO',	'COP',	'COR',	'COST',	'CPAY',	'CPB',	'CPRT',	'CPT',	'CRL',	'CRM',	'CRWD',	'CSCO',	'CSGP',	'CSX',	'CTAS',	'CTRA',	'CTSH',	'CTVA',	'CVS',	'CVX',	'CZR',	'D',	'DAL',	'DASH',	'DAY',	'DD',	'DDOG',	'DE',	'DECK',	'DELL',	'DG',	'DGX',	'DHI',	'DHR',	'DIS',	'DLR',	'DLTR',	'DOC',	'DOV',	'DOW',	'DPZ',	'DRI',	'DTE',	'DUK',	'DVA',	'DVN',	'DXCM',	'EA',	'EBAY',	'ECL',	'ED',	'EFX',	'EG',	'EIX',	'EL',	'ELV',	'EMN',	'EMR',	'ENPH',	'EOG',	'EPAM',	'EQIX',	'EQR',	'EQT',	'ERIE',	'ES',	'ESS',	'ETN',	'ETR',	'EVRG',	'EW',	'EXC',	'EXE',	'EXPD',	'EXPE',	'EXR',	'F',	'FANG',	'FAST',	'FCX',	'FDS',	'FDX',	'FE',	'FFIV',	'FI',	'FICO',	'FIS',	'FITB',	'FOX',	'FOXA',	'FRT',	'FSLR',	'FTNT',	'FTV',	'GD',	'GDDY',	'GE',	'GEHC',	'GEN',	'GEV',	'GILD',	'GIS',	'GL',	'GLW',	'GM',	'GNRC',			'GPC',	'GPN',	'GRMN',	'GS',	'GWW',	'HAL',	'HAS',	'HBAN',	'HCA',	'HD',	'HES',	'HIG',	'HII',	'HLT',	'HOLX',	'HON',	'HPE',	'HPQ',	'HRL',	'HSIC',	'HST',	'HSY',	'HUBB',	'HUM',	'HWM',	'IBM',	'ICE',	'IDXX',	'IEX',	'IFF',	'INCY',	'INTC',	'INTU',	'INVH',	'IP',	'IPG',	'IQV',	'IR',	'IRM',	'ISRG',	'IT',	'ITW',	'IVZ',	'J',	'JBHT',	'JBL',	'JCI',	'JKHY',	'JNJ',	'JPM',	'K',	'KDP',	'KEY',	'KEYS',	'KHC',	'KIM',	'KKR',	'KLAC',	'KMB',	'KMI',	'KMX',	'KO',	'KR',	'KVUE',	'L',	'LDOS',	'LEN',	'LH',	'LHX',	'LII',	'LIN',	'LKQ',	'LLY',	'LMT',	'LNT',	'LOW',	'LRCX',	'LULU',	'LUV',	'LVS',	'LW',	'LYB',	'LYV',	'MA',	'MAA',	'MAR',	'MAS',	'MCD',	'MCHP',	'MCK',	'MCO',	'MDLZ',	'MDT',	'MET',		'MGM',	'MHK',	'MKC',	'MKTX',	'MLM',	'MMC',	'MMM',	'MNST',	'MO',	'MOH',	'MOS',	'MPC',	'MPWR',	'MRK',	'MRNA',	'MS',	'MSCI',		'MSI',	'MTB',	'MTCH',	'MTD',	'MU',	'NCLH',	'NDAQ',	'NDSN',	'NEE',	'NEM',	'NFLX',	'NI',	'NKE',	'NOC',	'NOW',	'NRG',	'NSC',	'NTAP',	'NTRS',	'NUE',		'NVR',	'NWS',	'NWSA',	'NXPI',	'O',	'ODFL',	'OKE',	'OMC',	'ON',	'ORCL',	'ORLY',	'OTIS',	'OXY',	'PANW',	'PARA',	'PAYC',	'PAYX',	'PCAR',	'PCG',	'PEG',	'PEP',	'PFE',	'PFG',	'PG',	'PGR',	'PH',	'PHM',	'PKG',	'PLD',	'PLTR',	'PM',	'PNC',	'PNR',	'PNW',	'PODD',	'POOL',	'PPG',	'PPL',	'PRU',	'PSA',	'PSX',	'PTC',	'PWR',	'PYPL',	'QCOM',	'RCL',	'REG',	'REGN',	'RF',	'RJF',	'RL',	'RMD',	'ROK',	'ROL',	'ROP',	'ROST',	'RSG',	'RTX',	'RVTY',	'SBAC',	'SBUX',	'SCHW',	'SHW',	'SJM',	'SLB',	'SMCI',	'SNA',	'SNPS',	'SO',	'SOLV',	'SPG',	'SPGI',	'SRE',	'STE',	'STLD',	'STT',	'STX',	'STZ',	'SW',	'SWK',	'SWKS',	'SYF',	'SYK',	'SYY',	'T',	'TAP',	'TDG',	'TDY',	'TECH',	'TEL',	'TER',	'TFC',	'TGT',	'TJX',	'TKO',	'TMO',	'TMUS',	'TPL',	'TPR',	'TRGP',	'TRMB',	'TROW',	'TRV',	'TSCO',		'TSN',	'TT',	'TTWO',	'TXN',	'TXT',	'TYL',	'UAL',	'UBER',	'UDR',	'UHS',	'ULTA',	'UNH',	'UNP',	'UPS',	'URI',	'USB',	'V',	'VICI',	'VLO',	'VLTO',	'VMC',	'VRSK',	'VRSN',	'VRTX',	'VST',	'VTR',	'VTRS',	'VZ',	'WAB',	'WAT',	'WBA',	'WBD',	'WDAY',	'WDC',	'WEC',	'WELL',	'WFC',	'WM',	'WMB',	'WMT',	'WRB',	'WSM',	'WST',	'WTW',	'WY',	'WYNN',	'XEL',	'XOM',	'XYL',	'YUM',	'ZBH',	'ZBRA',	'ZTS','AAL',	'AAMRQ',	'ABI',	'ABS',	'ABX',	'ACKH',	'ACV',	'AET',	'AGC',	'AGN',	'AHM',	'AIT',	'AL',	'AM',	'AMH',	'AN',	'ANDW',	'ANV',	'AR',	'ARC',	'ARNC',	'AS',	'ASC',	'ASH',	'AT',	'ATI',	'AVP',	'AZA.A',	'BBI',	'BC',	'BCO',	'BCR',	'BDK',	'BEAM',	'BEV',	'BFI',	'BFO',	'BGG',	'BHGE',	'BHMSQ',	'BKB',	'BLL',	'BLS',	'BLY',	'BMET',	'BMS',	'BNI',	'BNL',	'BOAT',	'BOL',	'BT',	'BUD',	'CA',	'CAL',	'CAR',	'CBB',	'CBE',	'CBS',	'CCB', 'CCK',	'CCTYQ',	'CEN',	'CFL',	'CG',	'CGP',	'CHA',	'CHRS',	'CIN',	'CMA',	'CMB',	'CNG',	'CNW',	'COMS',	'COV',	'CPQ',	'CR',	'CRR',	'CSR',	'CTB',	'CTX',	'CYM',	'CYR',	'DALRQ',	'DCNAQ',	'DDS',	'DEC',	'DGN',	'DI',	'DIGI',	'DJ',	'DLX',	'DWD',	'DXC',	'EC',	'ECH',	'ECO',	'EFU',	'EKDKQ',	'ENRNQ',	'ENS',	'ETS',	'FBF',	'FBO',	'FCN',	'FDC',	'FG',	'FJ',	'FL',	'FLMIQ',	'FLTWQ',	'FMC',	'FMCC',	'FNMA',	'FTL.A',	'FWLT',	'G',	'GAPTQ',	'GAS',	'GDW',	'GFS.A',	'GIDL',	'GLD',	'GLK',	'GP',	'GPS',	'GPU',	'GR',	'GRA',	'GRN',	'GSX',	'GT',	'GTE',	'GWF',	'H',	'HDLM',	'HET',	'HI',	'HM',	'HNZ',	'HP',	'HPC',	'HPH',	'HRB',	'HRS',	'HSH',	'I',	'IKN',	'INCLF',	'INGR',	'ITT',	'JAVA',	'JCP',	'JH',	'JOS',	'JP',	'JWN',	'KATE',	'KBH',	'KM',	'KMG',	'KRB',	'KRI',	'KWP',	'LB',	'LDG',	'LDW.B',	'LLX',	'LNC',	'LOR',	'LPX',	'LSI',	'LUB',	'M',	'MAT',	'MAY',	'MCIC',	'MD',	'MDP',	'MDR',	'MEA',	'MEE',	'MEL',	'MER',	'MII',	'MIL',	'MKG',	'MNR',	'MOB',	'MRO',	'MST',	'MTLQQ',	'MWI',	'MWV',	'MYG',	'MZIAQ',	'NAE',	'NAV',	'NC',	'NCC',	'NLC',	'NMK',	'NOVL',	'NRTLQ',	'NSI',	'NSM',	'NWL',	'NYN',	'NYT',	'OAT',	'OM',	'OMX',	'ONE',	'ORX',	'OWENQ',	'PAC',	'PAS',	'PBI',	'PBY',	'PCH',	'PD',	'PDG',	'PEL',	'PET',	'PGL',	'PGN',	'PHA',	'PHB',	'PKI',	'PLL',	'PMI',	'PNU',	'PPW',	'PRD',	'PVN',	'PX',	'PZE',	'R',	'RAD',	'RAL',	'RBD',	'RBK',	'RDC',	'RDS.A',	'RLM',	'RML',	'RNB',	'ROH',	'RRD',	'RSHCQ',	'RTN',	'RYAN',	'RYC',	'RYI',	'S',	'SAF',	'SB',	'SCI',	'SFA',	'SFS',	'SGID',	'SGP',	'SHN',	'SIAL',	'SK',	'SMI',	'SMS',	'SNT',	'SRR',	'STI',	'STJ',	'STO',	'SUN',	'SVU',	'TA',	'TCOMA',	'TDM',	'TEK',	'TEN',	'TGNA',	'THC',	'THY',	'TIN',	'TKR',	'TLAB',	'TMC',	'TMC.A',	'TMK',	'TNB',	'TOY',	'TRB',	'TRW',	'TWX',	'TX',	'TXU',	'UAWGQ',	'UCC',	'UCL',	'UCM',	'UIS',	'UK',	'UMG',	'UN',	'UNM',	'USBC',	'USH',	'USHC',	'USS',	'UST',	'USW',	'UTX',	'VAT',	'VFC',	'VO',	'WAI',	'WB',	'WEN',	'WHR',	'WLA',	'WLL',	'WMX',	'WNDXQ',	'WOR',	'WWY',	'WYE',	'X',	'XRX',	'YRCW',	'CSE',	'BAY',	'GNT',	'EMC',	'NLV',	'WCOEQ',	'TUP',	'MTG',	'BMGCA',	'HFS',	'SEG',	'LU',	'UPR',	'RX',	'MBI',	'GDT',	'FRO',	'EHC',	'CFC',	'WAMUQ',	'SAI',	'APC',	'OI',	'MIR',	'CCU',	'HBOC',	'SNV',	'BBT',	'LEHMQ',	'BIG',	'SUB',	'MTL',	'NXTL',	'SEE',	'AFS.A',	'GTW',	'ASND',	'BSC',	'SLM',	'FMY',	'EDS',	'KSS',	'PVT',	'NGH',	'HCR',	'BMC',	'UPC',	'PSFT',	'SPLS',	'CCE',	'NCE',	'SWY',	'PCS',	'SLR',	'CPWR',	'SOTR',	'ASO',	'CTL',	'KSU',	'DPHIQ',	'PWJ',	'WLP',	'FPC',	'ODP',	'ADCT',	'AW',	'COC.B',	'LXK',	'TOS',	'GX',	'BBBY',	'SXCL',	'LEG',	'EP',	'CMVT',	'PTV',	'XLNX',	'QTRN',	'CTXS',	'MOLX',	'OK',	'AABA',	'RIG',	'NCR',	'YNR',	'BGEN',	'CNXT',	'HOG',	'TSG',	'LLTC',	'VRTS',	'ALTR',	'SAPE',	'SEBL',	'MXIM',	'APCC',	'CVG',	'MEDI',	'NVLS',	'SANM',	'TIF',	'MERQ',	'VSTNQ',	'BRCM',	'Q',	'JNS',	'CIT.A',	'VIAV',	'PALM',	'KSE',	'AV',	'DYN',	'KG',	'NBR',	'PWER',	'BVSN',	'HOT',	'FRX',	'CHIR',	'CPNLQ',	'RHI',	'ABKFQ',	'AYE',	'SBL',	'QLGC',	'VTSS',	'FLR',	'AMCC',	'NE',	'UVN',	'FTR',	'CE',	'FISV',	'PBG',	'MWW',	'ZION',	'JHF',	'COL',	'AWE',	'PMCS',	'FDO',	'ABC',	'CIEN',	'IGT',	'XL',	'IMNX',	'EOP',	'TE',	'HMA',	'GENZ',	'JNY',	'PCL',	'RATL',	'MI',	'FHN',	'APOL',	'BJS',	'NFB',	'SDS',	'ANTM',	'MON',	'RAI',	'AIV',	'SYMC',	'FII',	'MHS',	'ESRX',	'CMX',	'ETFC',	'ACS',	'MYL',	'HSP',	'SOV',	'FSH',	'CITGQ',	'LLL',	'FSL',	'ASN',	'CBSS',	'XTO',	'NOV',	'SHLD',	'WFT',	'VNO',	'MUR',	'CVH',	'PDCO',	'AMZN',	'GNW',	'SSP',	'VIAB',	'WFM',	'HAR',	'BRL',	'CHK',	'DF',	'SNDK',	'LM',	'EQ',	'JNPR',	'CBH',	'CNX',	'WIN',	'WYND',	'SII',	'WU',	'CELG',	'BTUUQ',	'IAC',	'STR',	'DTV',	'TEX',	'SE',	'ESV',	'VAR',	'HCBK',	'TEG',	'DDR',	'ANF',	'SUNEQ',	'PCP',	'DFS',	'GGP',	'ACAS',	'JEF',	'ANDV',	'TDC',	'NBL',	'NYX',	'JEC',	'TIE',	'POM',	'MTW',	'GME',	'RRC',	'GHC',	'TSS',	'CAM',	'HCP',	'SWN',	'LO',	'COG',	'AKS',	'SNI',	'PXD',	'FLS',	'PBCT',	'XRAY',	'CEPH',	'SRCL',	'LIFE',	'DNB',	'MFE',	'FLIR',	'SCG',	'DO',	'TWC',	'DNR',	'FTI',	'ATGE',	'RHT',	'CFN',	'ARG',	'CLF',	'MJN',	'URBN',	'DISCA',	'CERN',	'QEP',	'CVC',	'NFX',	'MMI',	'JOY',	'ANRZQ',	'BWA',	'PRGO',	'TRIP',	'WPX',	'FOSL',	'ALXN',	'ADT',	'KRFT',	'PETM',	'PVH',	'MAC',	'NLSN',	'KORS',	'ADS',	'FB',	'GMCR',	'NAVI',	'UAA',	'XEC',	'AMG',	'DISCK',	'MNK',	'LVLT',	'ENDP',	'HBI',	'SLG',	'QRVO',	'BXLT',	'CPGX',	'WRK',	'AAP',	'SIG',	'ATVI',	'CMCSK',	'FCPT',	'ILMN',	'CSRA',	'CCEP',	'CPRI',	'WLTW',	'CXO',	'UA',	'AYI',	'ALK',	'FBHS',	'COTY',	'EVHC',	'DISH',	'INFO',	'RE',	'DRE',	'BHF',	'DWDP',	'IPGP',	'NKTR',	'SIVB',	'ABMD',	'TWTR',	'HFC',	'FLT',	'WCG',	'FRC',	'TFX',	'NLOK',	'PEAK',	'VIAC',	'BIO',	'LUMN',	'CTLT',	'ETSY',	'VNT',	'PENN',	'OGN',	'BBWI',	'CDAY',	'SBNY',	'SEDG',	'AMTM']

stocklist = []
for t in ticker:
    try:
        contract = Stock(t, 'SMART', 'USD', primaryExchange='NASDAQ')
    except:
        contract = Stock(t, 'SMART', 'USD', primaryExchange='NYSE')

    bars = ib.reqHistoricalData(
        contract, endDateTime = '20260424 15:59:00', durationStr='1 D',
        barSizeSetting='1 day', whatToShow='TRADES', useRTH=True)

    df = util.df(bars)
    # df['date'] = df['date'].dt.tz_convert("America/New_York")
    if df is None:
        print(f"Ticker {t} is empty")
    else:
        df['actual_date'] = df['date']
        df['Ticker'] = t
        print(df.tail(1))
        stocklist.append(df)

#reset index to form one full, single table
stocklist=pd.concat(stocklist,ignore_index=True)
stocklist.reset_index(inplace=True,drop=True)  

#add dates and alter dtypes
stocklist['barCount'] = stocklist['barCount'].astype(float)
stocklist['MarketDate'] = stocklist['date']
stocklist['MarketTime'] = stocklist['date']
stocklist['RegistryDate'] = date.today()

#reorder columns
stocklist = stocklist[['MarketTime','open','high','low','close','volume','average','barCount','MarketDate','Ticker','RegistryDate']]
stocklist['Currency'] = 'USD'

#Remove date and replace by MarketTime
stocklist['MarketTime'] = stocklist['MarketDate']
stocklist = stocklist[['MarketTime','open','high','low','close','volume','average','barCount','MarketDate','Ticker','RegistryDate','Currency']]

#environment variable
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'c:\\Users\\Johnny\\Desktop\\Repo\\Projects\\financialdata-464612-483d8c5985f1.json'
client = bigquery.Client()

dataset_id = "PriceData"
table_id = "DailyPriceData"
table_ref = client.dataset(dataset_id).table(table_id)
table = client.get_table(table_ref)

job_config = bigquery.LoadJobConfig(
    # schema=schema, # Uncomment if you define a schema
    write_disposition="WRITE_APPEND",  # Overwrite table, or "WRITE_APPEND" to append
)

try:
    # Load DataFrame to BigQuery
    job = client.load_table_from_dataframe(stocklist, table_ref, job_config=job_config)
    job.result()  # Wait for the job to complete

    print(f"Loaded {job.output_rows} rows into {dataset_id}.{table_id}")

except Exception as e:
    print(f"Error loading data to BigQuery: {e}")

         date    open    high     low  close      volume  average  barCount  \
0  2026-04-24  373.55  382.76  370.73  376.3  45387602.0  376.121    364437   

  actual_date Ticker  
0  2026-04-24   TSLA  
         date    open    high     low   close       volume  average  barCount  \
0  2026-04-24  199.96  210.95  199.81  208.27  147838865.0  207.307    587566   

  actual_date Ticker  
0  2026-04-24   NVDA  
         date    open    high     low   close      volume  average  barCount  \
0  2026-04-24  272.79  273.06  269.65  271.06  18287115.0  270.852     96522   

  actual_date Ticker  
0  2026-04-24   AAPL  
         date    open    high     low   close      volume  average  barCount  \
0  2026-04-24  416.95  424.95  415.95  424.62  16066734.0   420.62    142520   

  actual_date Ticker  
0  2026-04-24   MSFT  
         date    open    high     low   close     volume  average  barCount  \
0  2026-04-24  337.59  343.69  334.05  342.32  8224253.0  339.826     48941   

  actual_date

C:\Users\Johnny\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 724 rows into PriceData.DailyPriceData
